# Spike 03 — English OCR with Chandra 2 (Datalab API)

**Goal:** Extract structured English text from the Tranquil City English PDF using Chandra 2 via the Datalab REST API.

**Time box:** 1 hour

**Question to answer:** Does Chandra produce well-structured Markdown (especially tables) from this English report?

**Done when:** `ocr_output/english/tranquil_en.md` contains readable text with tables preserved as Markdown.

---

### How the API actually works (async)

```
POST /api/v1/convert  ← send the PDF
      ↓
{ "request_check_url": "https://..." }   ← get a polling URL back

GET request_check_url  (every 2 seconds)
      ↓
{ "status": "pending" }   ← not done yet, keep polling
{ "status": "complete", "markdown": "# Full extracted text..." }  ← done
```

No GPU. No RAM. No image extraction. One PDF in → Markdown out.

### API Key Setup
1. Go to [datalab.to](https://www.datalab.to) → sign up → get **\$5 free credits**
2. Add to `.env`: `CHANDRA_API_KEY=your_key_here`

## Step 1 — Imports & Setup

In [ ]:
import requests
import time
from dotenv import load_dotenv
from pathlib import Path
import os

load_dotenv(dotenv_path="../.env")

API_KEY     = os.getenv("CHANDRA_API_KEY")
CONVERT_URL = "https://www.datalab.to/api/v1/convert"
HEADERS     = {"X-API-Key": API_KEY}   # ← X-API-Key, not Authorization: Bearer

if not API_KEY:
    raise ValueError("CHANDRA_API_KEY not found in .env file")

print("✅ Datalab client ready")
print(f"   Endpoint : {CONVERT_URL}")
print(f"   Auth     : X-API-Key: {API_KEY[:8]}...")

## Step 2 — Core Functions

In [ ]:
def submit_pdf(pdf_path: str, output_format: str = "markdown") -> str:
    """
    Submit a PDF to Datalab and return the check URL for polling.
    This is non-blocking — it returns immediately with a URL to check later.
    """
    with open(pdf_path, "rb") as f:
        response = requests.post(
            CONVERT_URL,
            files   = {"file": (Path(pdf_path).name, f, "application/pdf")},
            data    = {"output_format": output_format},
            headers = HEADERS,
            timeout = 30
        )

    if response.status_code != 200:
        raise RuntimeError(f"Submit failed {response.status_code}: {response.text[:300]}")

    data = response.json()
    print(f"   Submitted. Check URL: {data['request_check_url']}")
    return data["request_check_url"]


def poll_until_done(check_url: str, poll_interval: int = 2, max_wait: int = 300) -> str:
    """
    Poll the check URL every `poll_interval` seconds until status == 'complete'.
    Returns the extracted markdown text.

    max_wait: maximum seconds to wait before giving up (default 5 minutes)
    """
    waited = 0
    while waited < max_wait:
        response = requests.get(check_url, headers=HEADERS, timeout=15)
        result   = response.json()
        status   = result.get("status", "unknown")

        print(f"   Status: {status}  ({waited}s elapsed)", end="\r")

        if status == "complete":
            print(f"\n   ✅ Done in {waited}s")
            return result["markdown"]

        if status == "error":
            raise RuntimeError(f"API returned error: {result}")

        time.sleep(poll_interval)
        waited += poll_interval

    raise TimeoutError(f"Job did not complete within {max_wait}s")


def convert_pdf(pdf_path: str, output_format: str = "markdown") -> str:
    """
    Full pipeline: submit PDF → poll → return markdown.
    One call to rule them all.
    """
    print(f"Submitting: {Path(pdf_path).name}")
    check_url = submit_pdf(pdf_path, output_format)
    return poll_until_done(check_url)


print("✅ Functions defined: submit_pdf(), poll_until_done(), convert_pdf()")

## Step 3 — Convert the English PDF

In [ ]:
import fitz  # PyMuPDF — already installed from Spike 01

OUTPUT_DIR   = Path("../ocr_output/english")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FULL_PDF     = Path("../data/pdfs/tranquil_en.pdf")
SUBSET_PDF   = Path("../data/pdfs/tranquil_en_subset.pdf")  # 14-page extract
OUT_PATH     = OUTPUT_DIR / "tranquil_en.md"
PAGE_LIMIT   = 14   # match Spike 02 subset size

# ── Step A: Extract first N pages into a smaller PDF ────────────
if SUBSET_PDF.exists():
    print(f"⏭  Subset PDF already exists — {SUBSET_PDF.stat().st_size // 1024} KB")
else:
    doc    = fitz.open(str(FULL_PDF))
    subset = fitz.open()                         # blank PDF
    subset.insert_pdf(doc, from_page=0, to_page=PAGE_LIMIT - 1)
    subset.save(str(SUBSET_PDF))
    doc.close(); subset.close()

    orig_kb   = FULL_PDF.stat().st_size   // 1024
    subset_kb = SUBSET_PDF.stat().st_size // 1024
    print(f"✅ Subset created: {PAGE_LIMIT} pages")
    print(f"   Original : {orig_kb:,} KB")
    print(f"   Subset   : {subset_kb:,} KB  ({subset_kb/orig_kb*100:.0f}% of original)")

# ── Step B: Submit subset to Datalab ────────────────────────────
if OUT_PATH.exists():
    print(f"\n⏭  Already converted — {OUT_PATH.stat().st_size // 1024} KB")
    print(f"   Delete {OUT_PATH.name} to re-run")
else:
    markdown = convert_pdf(str(SUBSET_PDF))
    OUT_PATH.write_text(markdown, encoding="utf-8")
    print(f"\n💾 Saved → {OUT_PATH.resolve()}")
    print(f"   Size : {len(markdown):,} characters")

## Step 4 — Inspect the Output

In [ ]:
content = OUT_PATH.read_text(encoding="utf-8")

print(f"File     : {OUT_PATH.name}")
print(f"Size     : {len(content):,} characters")
print(f"Has tables (|) : {'✅ Yes' if '|' in content else '❌ No'}")
print(f"Has headings   : {'✅ Yes' if '#' in content else '❌ No'}")
print()

# Show a section from the middle of the document (more likely to have data)
mid = len(content) // 2
print("=== MIDDLE SECTION PREVIEW ===")
print(content[mid:mid + 2000])

## Step 5 — Spike Summary

In [ ]:
content = OUT_PATH.read_text(encoding="utf-8") if OUT_PATH.exists() else ""

print("=" * 50)
print("SPIKE 03 — RESULTS")
print("=" * 50)
print(f"OCR engine     : Chandra 2 (Datalab Cloud API)")
print(f"Input          : tranquil_en.pdf (full document)")
print(f"Output         : tranquil_en.md")
print(f"Total chars    : {len(content):,}")
print(f"Has tables     : {'✅ Yes' if '|' in content else '❌ No'}")
print(f"Has headings   : {'✅ Yes' if '#' in content else '❌ No'}")
print()

if len(content) > 1000:
    print("✅ SPIKE PASSED — English OCR output ready for Spike 04 (PageIndex)")
    print(f"   File: {OUT_PATH.resolve()}")
else:
    print("❌ SPIKE FAILED — output is empty or too short")
    print("   Check: Is the API key correct? Did the PDF download in Spike 01?")